# FHVHV Trip Data — Raw Tables Exploration

**Goal:** Explore the parquet trip data and design **raw tables** separated by **context** (logical domains) for a clean raw layer.

- **Data source:** NYC TLC FHVHV trip records (e.g. `fhvhv_tripdata_2026-01.parquet`)
- **Output:** Context-based raw table definitions and, optionally, written parquet files per context

In [2]:
import pandas as pd
import duckdb
from pathlib import Path

# Parquet path: use parent dir if file lives next to repo, or set to your path
PARQUET_PATH = Path("../fhvhv_tripdata_2026-01.parquet")
assert PARQUET_PATH.exists(), f"Parquet not found at {PARQUET_PATH}. Adjust PARQUET_PATH."

## 1. Load and inspect schema

Using DuckDB to avoid loading the full ~21M rows into memory; we'll sample for exploration.

In [12]:
con = duckdb.connect()

# Row count
n_rows = con.execute(f"SELECT count(*) FROM read_parquet('{PARQUET_PATH}')").fetchone()[0]
print(f"Total rows: {n_rows:,}")

# Schema (column names + types)
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PARQUET_PATH}')").fetchall()
print("\nSchema:")

# test = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PARQUET_PATH}')").df()
# test

for col_name, col_type, null, key, default, extra in schema:
    print(f"  {col_name}: {col_type} - has null {null}")

Total rows: 20,940,373

Schema:
  hvfhs_license_num: VARCHAR - has null YES
  dispatching_base_num: VARCHAR - has null YES
  originating_base_num: VARCHAR - has null YES
  request_datetime: TIMESTAMP - has null YES
  on_scene_datetime: TIMESTAMP - has null YES
  pickup_datetime: TIMESTAMP - has null YES
  dropoff_datetime: TIMESTAMP - has null YES
  PULocationID: INTEGER - has null YES
  DOLocationID: INTEGER - has null YES
  trip_miles: DOUBLE - has null YES
  trip_time: BIGINT - has null YES
  base_passenger_fare: DOUBLE - has null YES
  tolls: DOUBLE - has null YES
  bcf: DOUBLE - has null YES
  sales_tax: DOUBLE - has null YES
  congestion_surcharge: DOUBLE - has null YES
  airport_fee: DOUBLE - has null YES
  tips: DOUBLE - has null YES
  driver_pay: DOUBLE - has null YES
  shared_request_flag: VARCHAR - has null YES
  shared_match_flag: VARCHAR - has null YES
  access_a_ride_flag: VARCHAR - has null YES
  wav_request_flag: VARCHAR - has null YES
  wav_match_flag: VARCHAR - has nu

In [13]:
# Sample for exploration (e.g. 50k rows)
SAMPLE_SIZE = 50_000
df = con.execute(f"""
    SELECT * FROM read_parquet('{PARQUET_PATH}')
    ORDER BY random()
    LIMIT {SAMPLE_SIZE}
""").df()

print(f"Sample shape: {df.shape}")
df.head(10)

Sample shape: (50000, 25)


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
0,HV0005,B03406,None,2026-01-27 13:13:38,2026-01-27 13:15:32,2026-01-27 13:16:19,2026-01-27 13:23:51,82,196,0.967,...,0.00,0.0,0.00,6.16,N,N,N,N,N,0.0
1,HV0005,B03406,None,2026-01-06 16:36:55,2026-01-06 16:39:03,2026-01-06 16:39:37,2026-01-06 17:06:48,152,13,8.213,...,2.75,0.0,0.00,28.17,N,N,N,N,N,0.0
2,HV0003,B03404,B03404,2026-01-24 19:58:10,2026-01-24 19:58:58,2026-01-24 19:59:31,2026-01-24 20:09:47,249,79,1.130,...,2.75,0.0,0.00,8.16,N,N,N,N,N,1.5
3,HV0003,B03404,B03404,2026-01-14 12:47:10,2026-01-14 12:53:57,2026-01-14 12:54:39,2026-01-14 13:03:15,91,89,1.300,...,0.00,0.0,0.00,11.95,N,N,N,N,Y,0.0
4,HV0005,B03406,None,2026-01-06 06:20:01,2026-01-06 06:13:56,2026-01-06 06:16:35,2026-01-06 06:42:27,181,39,5.318,...,0.00,0.0,6.19,26.79,N,N,N,N,Y,0.0
5,HV0003,B03404,B03404,2026-01-23 16:11:37,2026-01-23 16:14:23,2026-01-23 16:14:47,2026-01-23 16:31:17,254,78,3.330,...,0.00,0.0,0.00,15.00,N,N,N,N,N,0.0
6,HV0005,B03406,None,2026-01-14 03:57:43,2026-01-14 04:03:51,2026-01-14 04:04:51,2026-01-14 04:09:36,76,76,0.910,...,0.00,0.0,0.00,4.27,N,N,N,N,N,0.0
7,HV0003,B03404,B03404,2026-01-24 15:33:53,2026-01-24 15:34:37,2026-01-24 15:35:00,2026-01-24 15:45:02,7,7,0.930,...,0.00,0.0,0.00,7.76,N,N,N,N,N,0.0
8,HV0003,B03404,B03404,2026-01-17 00:48:02,2026-01-17 00:54:15,2026-01-17 00:56:15,2026-01-17 01:35:15,186,39,21.110,...,2.75,0.0,0.00,51.90,N,N,N,N,N,1.5
9,HV0005,B03406,None,2026-01-31 17:59:26,2026-01-31 18:04:02,2026-01-31 18:04:05,2026-01-31 18:16:30,169,47,1.297,...,0.00,0.0,0.00,11.45,N,N,N,N,Y,0.0


## 2. Column overview and nulls

Understand completeness and types per column to assign contexts.

In [14]:
# Null counts and dtypes
info = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().sum() / len(df) * 100).round(2),
})
info

,dtype,null_count,null_pct
hvfhs_license_num,object,0,0.00
dispatching_base_num,object,0,0.00
originating_base_num,object,13685,27.37
request_datetime,datetime64[us],0,0.00
on_scene_datetime,datetime64[us],0,0.00
pickup_datetime,datetime64[us],0,0.00
dropoff_datetime,datetime64[us],0,0.00
PULocationID,int32,0,0.00
DOLocationID,int32,0,0.00
trip_miles,float64,0,0.00


## 3. Define contexts → raw tables

Group columns by **business context** so each raw table has a single responsibility. We need a **trip key** to join them; we'll use a 0-based row index as `trip_id` (or you can use a composite of timestamps + base later).

| Context | Description | Columns |
|---------|-------------|---------|
| **Dispatch / base** | Who dispatched, which base | hvfhs_license_num, dispatching_base_num, originating_base_num |
| **Trip / time** | When and where | request_datetime, on_scene_datetime, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, trip_miles, trip_time |
| **Fare / payment** | Money components | base_passenger_fare, tolls, bcf, sales_tax, congestion_surcharge, airport_fee, tips, driver_pay, cbd_congestion_fee |
| **Request flags** | Shared, Access-a-Ride, WAV | shared_request_flag, shared_match_flag, access_a_ride_flag, wav_request_flag, wav_match_flag |

In [15]:
# Raw table column groups (contexts)
RAW_TABLE_CONTEXTS = {
    "raw_dispatch_base": [
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
    ],
    "raw_trip_time_location": [
        "request_datetime",
        "on_scene_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
    ],
    "raw_fare_payment": [
        "base_passenger_fare",
        "tolls",
        "bcf",
        "sales_tax",
        "congestion_surcharge",
        "airport_fee",
        "tips",
        "driver_pay",
        "cbd_congestion_fee",
    ],
    "raw_request_flags": [
        "shared_request_flag",
        "shared_match_flag",
        "access_a_ride_flag",
        "wav_request_flag",
        "wav_match_flag",
    ],
}

# Trip key column (added to each raw table for joins)
TRIP_KEY = "trip_id"

# Partition column: processing date in yyyyMMdd (for partitioning raw outputs)
PARTITION_COL = "processed_date"

# Validate: all columns exist and are disjoint (except trip_id)
all_cols = []
for ctx, cols in RAW_TABLE_CONTEXTS.items():
    for c in cols:
        assert c in df.columns, f"Missing column: {c}"
        all_cols.append(c)
assert len(all_cols) == len(set(all_cols)), "Duplicate column across contexts"
assert len(all_cols) == len(df.columns), f"Contexts cover {len(all_cols)} cols, schema has {len(df.columns)}"
print("All schema columns assigned to exactly one raw table context.")

All schema columns assigned to exactly one raw table context.


## 4. Build raw tables (with `trip_id` and `processed_date`)

Add a 0-based `trip_id` so we can join the context tables back together, and a `processed_date` partition column in **yyyyMMdd** format (e.g. `20260308`). On full data, generate `trip_id` in a deterministic way (e.g. row number in a single read).

In [16]:
from datetime import datetime

# Add trip_id and partition column processed_date (yyyyMMdd)
df_with_id = df.reset_index(drop=True).reset_index().rename(columns={"index": TRIP_KEY})
df_with_id[PARTITION_COL] = datetime.now().strftime("%Y%m%d")

raw_tables = {}
for name, cols in RAW_TABLE_CONTEXTS.items():
    raw_tables[name] = df_with_id[[TRIP_KEY, PARTITION_COL] + cols].copy()

for name, tbl in raw_tables.items():
    print(f"{name}: {tbl.shape}")

raw_dispatch_base: (50000, 4)
raw_trip_time_location: (50000, 9)
raw_fare_payment: (50000, 10)
raw_request_flags: (50000, 6)


In [17]:
# Preview each raw table
for name, tbl in raw_tables.items():
    print(f"\n--- {name} ---")
    display(tbl.head())


--- raw_dispatch_base ---


,trip_id,hvfhs_license_num,dispatching_base_num,originating_base_num
0,0,HV0005,B03406,None
1,1,HV0005,B03406,None
2,2,HV0003,B03404,B03404
3,3,HV0003,B03404,B03404
4,4,HV0005,B03406,None



--- raw_trip_time_location ---


,trip_id,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time
0,0,2026-01-27 13:13:38,2026-01-27 13:15:32,2026-01-27 13:16:19,2026-01-27 13:23:51,82,196,0.967,452
1,1,2026-01-06 16:36:55,2026-01-06 16:39:03,2026-01-06 16:39:37,2026-01-06 17:06:48,152,13,8.213,1631
2,2,2026-01-24 19:58:10,2026-01-24 19:58:58,2026-01-24 19:59:31,2026-01-24 20:09:47,249,79,1.130,616
3,3,2026-01-14 12:47:10,2026-01-14 12:53:57,2026-01-14 12:54:39,2026-01-14 13:03:15,91,89,1.300,516
4,4,2026-01-06 06:20:01,2026-01-06 06:13:56,2026-01-06 06:16:35,2026-01-06 06:42:27,181,39,5.318,1552



--- raw_fare_payment ---


,trip_id,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee
0,0,11.66,0.0,0.29,1.03,0.00,0.0,0.00,6.16,0.0
1,1,35.17,0.0,0.88,3.12,2.75,0.0,0.00,28.17,0.0
2,2,17.30,0.0,0.37,1.44,2.75,0.0,0.00,8.16,1.5
3,3,9.83,0.0,0.24,0.87,0.00,0.0,0.00,11.95,0.0
4,4,27.85,0.0,0.68,2.41,0.00,0.0,6.19,26.79,0.0



--- raw_request_flags ---


,trip_id,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,0,N,N,N,N,N
1,1,N,N,N,N,N
2,2,N,N,N,N,N
3,3,N,N,N,N,Y
4,4,N,N,N,N,Y


## 5. Write raw tables to Parquet (optional)

Uncomment and set `OUTPUT_DIR` to persist raw tables as separate parquet files. For full data, run the same logic in DuckDB and write from there to avoid OOM.

In [18]:
OUTPUT_DIR = Path("data/raw")  # e.g. create and use this
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, tbl in raw_tables.items():
    out_path = OUTPUT_DIR / f"{name}.parquet"
    tbl.to_parquet(out_path, index=False)
    print(f"Wrote {out_path}")

Wrote data/raw/raw_dispatch_base.parquet
Wrote data/raw/raw_trip_time_location.parquet
Wrote data/raw/raw_fare_payment.parquet
Wrote data/raw/raw_request_flags.parquet


## 6. Full-data raw tables via DuckDB (scalable)

To build raw tables from the **full** parquet without loading it all into pandas, use DuckDB to project columns + `row_number()` as `trip_id` and write each context to its own parquet.

In [19]:
# Example: write one raw table from full parquet (DuckDB)
# Uncomment and set OUTPUT_DIR to run on full dataset.
# processed_date is partition column in yyyyMMdd (DuckDB: strftime(current_date, '%Y%m%d'))

OUTPUT_DIR = Path("data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processed_date_expr = "strftime(current_date, '%Y%m%d') AS processed_date"
dispatch_cols = ", ".join(RAW_TABLE_CONTEXTS["raw_dispatch_base"])
con.execute(f"""
    COPY (
        SELECT row_number() OVER () - 1 AS trip_id, {processed_date_expr}, {dispatch_cols}
        FROM read_parquet('{PARQUET_PATH}')
    ) TO '{OUTPUT_DIR / "raw_dispatch_base.parquet"}' (FORMAT PARQUET)
""")
# Repeat for raw_trip_time_location, raw_fare_payment, raw_request_flags (each with trip_id, processed_date, then context cols)

In [20]:
# Rejoin raw tables on trip_id (sanity check)
rejoined = (
    raw_tables["raw_dispatch_base"]
    .merge(raw_tables["raw_trip_time_location"], on=TRIP_KEY, how="inner")
    .merge(raw_tables["raw_fare_payment"], on=TRIP_KEY, how="inner")
    .merge(raw_tables["raw_request_flags"], on=TRIP_KEY, how="inner")
)
print("Rejoined shape:", rejoined.shape)
print("Columns match original:", set(rejoined.columns) == set(df_with_id.columns))
rejoined.head()

Rejoined shape: (50000, 26)
Columns match original: True


,trip_id,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,...,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,0,HV0005,B03406,None,2026-01-27 13:13:38,2026-01-27 13:15:32,2026-01-27 13:16:19,2026-01-27 13:23:51,82,196,...,0.00,0.0,0.00,6.16,0.0,N,N,N,N,N
1,1,HV0005,B03406,None,2026-01-06 16:36:55,2026-01-06 16:39:03,2026-01-06 16:39:37,2026-01-06 17:06:48,152,13,...,2.75,0.0,0.00,28.17,0.0,N,N,N,N,N
2,2,HV0003,B03404,B03404,2026-01-24 19:58:10,2026-01-24 19:58:58,2026-01-24 19:59:31,2026-01-24 20:09:47,249,79,...,2.75,0.0,0.00,8.16,1.5,N,N,N,N,N
3,3,HV0003,B03404,B03404,2026-01-14 12:47:10,2026-01-14 12:53:57,2026-01-14 12:54:39,2026-01-14 13:03:15,91,89,...,0.00,0.0,0.00,11.95,0.0,N,N,N,N,Y
4,4,HV0005,B03406,None,2026-01-06 06:20:01,2026-01-06 06:13:56,2026-01-06 06:16:35,2026-01-06 06:42:27,181,39,...,0.00,0.0,6.19,26.79,0.0,N,N,N,N,Y
